In [1]:
!pip install matplotlib
!pip install scipy

In [2]:
!sudo yum install -y java-11-amazon-corretto-headless

Last metadata expiration check: 0:36:51 ago on Sun Jul 26 08:44:47 2026.
Package java-11-amazon-corretto-headless-1:11.0.31+11-1.amzn2023.x86_64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!


In [1]:
import os
import glob

# Search for the installed Java directory
java_paths = glob.glob('/usr/lib/jvm/java-11*')

if java_paths:
    # Dynamically set the environment variable to the found path
    os.environ["JAVA_HOME"] = java_paths[0]
    print(f"JAVA_HOME successfully set to: {os.environ['JAVA_HOME']}")
else:
    print("Java not found. Did the yum install command work?")

JAVA_HOME successfully set to: /usr/lib/jvm/java-11-amazon-corretto.x86_64


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import time

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, input_file_name, countDistinct, desc, round, avg
from pyspark.sql.functions import date_format
import pyspark.sql.functions as F
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("DAT204M-EDA-Sagemaker")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .config("spark.driver.memory", "16g")
    .config("spark.executor.memory", "24g")
    .config("spark.executor.memoryOverhead", "6g") # up from 4g
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.local.dir", "/home/ec2-user/SageMaker/spark-tmp")
    .getOrCreate()
)



print("Spark ready:", spark.version)

26/07/26 10:59:01 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
Spark ready: 3.3.0


In [20]:
DATA_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"
df = spark.read.parquet(DATA_PATH)

In [21]:
# Enable clean HTML formatting for Spark DataFrames
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [22]:
from pyspark.sql import functions as F

In [23]:
features_df = (
    df.groupBy("product_id")
      .agg(
          # Event counts
          F.sum(F.when(F.col("event_id") == "item_view", 1).otherwise(0)).alias("views"),
          F.sum(F.when(F.col("event_id") == "item_like", 1).otherwise(0)).alias("likes"),
          F.sum(F.when(F.col("event_id") == "item_add_to_cart_tap", 1).otherwise(0)).alias("cart"),
          F.sum(F.when(F.col("event_id") == "offer_make", 1).otherwise(0)).alias("offers"),
          F.sum(F.when(F.col("event_id") == "buy_start", 1).otherwise(0)).alias("buy_start"),
          F.sum(F.when(F.col("event_id") == "buy_comp", 1).otherwise(0)).alias("buy_comp"),

          # Engagement
          F.countDistinct("user_id").alias("unique_users"),
          F.countDistinct("session_id").alias("unique_sessions"),

          # Price
          F.coalesce(F.avg("price"), F.lit(0.0)).alias("avg_price"),

          # Metadata
          F.coalesce(
              F.first("brand_name", ignorenulls=True),
              F.lit("Unknown")
          ).alias("brand_name"),

          # Category path
          F.concat_ws(
              " > ",
              F.coalesce(F.first("c0_name", ignorenulls=True), F.lit("Unknown")),
              F.coalesce(F.first("c1_name", ignorenulls=True), F.lit("Unknown")),
              F.coalesce(F.first("c2_name", ignorenulls=True), F.lit("Unknown"))
          ).alias("category_path"),

          # Condition
          F.sum(F.when(F.col("item_condition_name") == "Good", 1).otherwise(0)).alias("cond_good"),
          F.sum(F.when(F.col("item_condition_name") == "New", 1).otherwise(0)).alias("cond_new"),
          F.sum(F.when(F.col("item_condition_name") == "Like new", 1).otherwise(0)).alias("cond_like_new"),
          F.sum(F.when(F.col("item_condition_name") == "Fair", 1).otherwise(0)).alias("cond_fair"),
          F.sum(F.when(F.col("item_condition_name") == "Poor", 1).otherwise(0)).alias("cond_poor"),
          F.sum(
                F.when(
                    F.col("item_condition_name").isNull() |
                    (F.trim(F.col("item_condition_name")) == ""),
                    1
                ).otherwise(0)
            ).alias("cond_unknown")
      )
)

# Final safety net
features_df = features_df.fillna({
    "views": 0,
    "likes": 0,
    "cart": 0,
    "offers": 0,
    "buy_start": 0,
    "buy_comp": 0,
    "unique_users": 0,
    "unique_sessions": 0,
    "avg_price": 0.0,
    "brand_name": "Unknown",
    "category_path": "Unknown",
    "cond_good": 0,
    "cond_new": 0,
    "cond_like_new": 0,
    "cond_fair": 0,
    "cond_poor": 0,
    "cond_unknown": 0
})

log1p() was applied to highly skewed numerical features (e.g., views, likes, users, sessions, and price) to reduce the influence of extreme values and compress the range of the data. This helps K-Means form more balanced clusters, as the algorithm is sensitive to large differences in feature magnitudes. The log1p() function (log(1+x)) is used instead of log(x) because it safely handles zero values.

In [24]:
features_df = (
    features_df
    .withColumn("log_views", F.log1p("views"))
    .withColumn("log_likes", F.log1p("likes"))
    .withColumn("log_cart", F.log1p("cart"))
    .withColumn("log_offers", F.log1p("offers"))
    .withColumn("log_buy_start", F.log1p("buy_start"))
    .withColumn("log_buy_comp", F.log1p("buy_comp"))
    .withColumn("log_users", F.log1p("unique_users"))
    .withColumn("log_sessions", F.log1p("unique_sessions"))
    .withColumn("log_price", F.log1p("avg_price"))
    .withColumn("log_cond_good", F.log1p("cond_good"))
    .withColumn("log_cond_new", F.log1p("cond_new"))
    .withColumn("log_cond_like_new", F.log1p("cond_like_new"))
    .withColumn("log_cond_fair", F.log1p("cond_fair"))
    .withColumn("log_cond_poor", F.log1p("cond_poor"))
    .withColumn("log_cond_unknown", F.log1p("cond_unknown"))
)

In [25]:
s3_path = "s3a://dat204m-project-g3/feature_engineered_full"

features_df = features_df.repartition(4)
features_df.write \
    .mode("overwrite") \
    .parquet(s3_path)